# CropLens AI - Exploratory Data Analysis

**Project:** CropLens AI: APMC Market Intelligence, Supply Shock Detection and Procurement Intelligence Platform  
**Dataset:** `data/processed/features_master.parquet`  
**Records:** 38,355 rows × 53 columns | **Period:** 2019–2025  
**Target Variable:** `modal_price` (Rupees per Quintal)

---

## Table of Contents

1. [Import Libraries](#1-import-libraries)
2. [Load Dataset](#2-load-dataset)
3. [Dataset Overview](#3-dataset-overview)
4. [Unique Categorical Values](#4-unique-categorical-values)
5. [Missing Value Analysis](#5-missing-value-analysis)
6. [Duplicate Analysis](#6-duplicate-analysis)
7. [Summary Statistics](#7-summary-statistics)
8. [Target Variable Analysis (modal_price)](#8-target-variable-analysis-modal_price)
9. [Commodity Analysis](#9-commodity-analysis)
10. [Mandi Analysis](#10-mandi-analysis)
11. [Price Volatility and Spread](#11-price-volatility-and-spread)
12. [Arrival Volume Analysis](#12-arrival-volume-analysis)
13. [Weather Analysis](#13-weather-analysis)
14. [Heatwave Analysis by Crop](#14-heatwave-analysis-by-crop)
15. [Satellite NDVI Analysis](#15-satellite-ndvi-analysis)
16. [Yearly and Monthly Trends](#16-yearly-and-monthly-trends)
17. [Festival Period Analysis](#17-festival-period-analysis)
18. [Validation of Key Features](#18-validation-of-key-features)
19. [Correlation Analysis](#19-correlation-analysis)
20. [Important Features for Modeling](#20-important-features-for-modeling)
21. [Practical Takeaways](#21-practical-takeaways)
22. [Final Summary](#22-final-summary)

<a id="1-import-libraries"></a>
## 1. Import Libraries

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Set simple plot style
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

print('Libraries loaded successfully.')

### Plotting Functions

Simple reusable helper functions for charts.

In [ ]:
def plot_histogram(df, col, title, xlabel, color='steelblue', bins=40):
    """Plot a histogram for a numeric column."""
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(df[col].dropna(), bins=bins, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Frequency')
    ax.grid(axis='y', alpha=0.4)
    plt.tight_layout()
    plt.show()


def plot_boxplot(df, x, y, title, xlabel, ylabel, palette='Set2', figsize=(8, 4.5)):
    """Plot a boxplot grouped by category."""
    fig, ax = plt.subplots(figsize=figsize)
    sns.boxplot(data=df, x=x, y=y, hue=x, palette=palette, legend=False, ax=ax, fliersize=3)
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.4)
    plt.tight_layout()
    plt.show()


def plot_line(df, x, y, hue, title, xlabel, ylabel, figsize=(11, 4.5)):
    """Plot a line chart for time series data."""
    fig, ax = plt.subplots(figsize=figsize)
    sns.lineplot(data=df, x=x, y=y, hue=hue, ax=ax, linewidth=2)
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.4)
    ax.legend(title=hue, bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


def plot_bar(labels, values, title, xlabel, ylabel, color='steelblue', figsize=(8, 4), fmt='{:,.0f}'):
    """Plot a bar chart with numbers on top of bars."""
    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar(labels, values, color=color, edgecolor='white', alpha=0.9)
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.4)
    
    for bar in bars:
        height = bar.get_height()
        if not np.isnan(height):
            ax.annotate(fmt.format(height),
                        xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.show()


def plot_scatter(df, x, y, hue, title, xlabel, ylabel, alpha=0.4, sample=5000, figsize=(8, 4.5)):
    """Plot a scatter chart using a sample of points."""
    sample_df = df.sample(n=min(sample, len(df)), random_state=42)
    fig, ax = plt.subplots(figsize=figsize)
    sns.scatterplot(data=sample_df, x=x, y=y, hue=hue, alpha=alpha, ax=ax)
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.4)
    plt.tight_layout()
    plt.show()


print('Plotting functions ready.')

<a id="2-load-dataset"></a>
## 2. Load Dataset

In [ ]:
# File path resolution supporting execution from project root or notebooks folder
if os.path.exists(os.path.join('data', 'processed', 'features_master.parquet')):
    DATA_PATH = os.path.join('data', 'processed', 'features_master.parquet')
elif os.path.exists(os.path.join('..', 'data', 'processed', 'features_master.parquet')):
    DATA_PATH = os.path.join('..', 'data', 'processed', 'features_master.parquet')
else:
    raise FileNotFoundError('features_master.parquet dataset file not found.')

df = pd.read_parquet(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])

print(f'Master Dataset successfully loaded from: {DATA_PATH}')
print(f'Dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

<a id="3-dataset-overview--memory-footprint"></a>
## 3. Dataset Overview & Memory Footprint

In [ ]:
print('=== Dataset Summary ===')
print(f'Rows (Observations) : {df.shape[0]:,}')
print(f'Columns (Features)  : {df.shape[1]}')
print(f'Date Range          : {df["date"].min().date()} to {df["date"].max().date()}')
print()
print('=== Detailed Memory Footprint ===')
df.info(memory_usage='deep')

### Observation

- The master dataset contains **38,355 rows** and **53 columns**, spanning 7 years (2019 to 2025).
- Total memory footprint is approximately **15.4 MB**. The dataset is relatively small and can comfortably fit into memory, supporting efficient analysis and model training.
- Column data types are structured cleanly: 37 float64, 7 int64, 8 object/string columns, and 1 datetime column.

<a id="4-categorical-diversity--unique-value-summary"></a>
## 4. Categorical Diversity & Unique Value Summary

In [ ]:
cat_cols = ['state', 'district', 'market', 'commodity', 'variety', 'harvest_season_type', 'festival_name']
unique_summary = pd.DataFrame({
    'Categorical Column': cat_cols,
    'Unique Count': [df[c].nunique() for c in cat_cols],
    'Categories Present': [list(df[c].dropna().unique()) for c in cat_cols]
})
unique_summary

### Observation

- The dataset encompasses 3 baseline commodities (Onion, Potato, Tomato) across 5 APMC mandis in 5 agricultural districts.
- Categorical metadata covers major harvest season types (Kharif, Rabi, Zaid) and festival windows.
- The data structure allows comparing market behavior across both major consumption hubs (Delhi) and primary production regions (Nashik, Agra, Kolar).

<a id="5-missing-value-analysis--time-series-warm-up-nans"></a>
## 5. Missing Value Analysis & Time-Series Warm-up NaNs

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing Percent (%)': missing_pct
}).query('`Missing Count` > 0').sort_values('Missing Count', ascending=False)

if missing_df.empty:
    print('No missing values found in the dataset.')
else:
    print(f'Columns with missing values ({len(missing_df)} total):')
    print(missing_df)

    plot_bar(missing_df.index, missing_df['Missing Percent (%)'],
             'Missing Value Percentage by Column',
             'Feature Column', 'Missing (%)', color='coral', fmt='{:.1f}%')

### Observation

- All primary source columns (raw prices, arrival volume, weather, satellite NDVI) have zero missing values, confirming solid source data quality.
- Missing values exist only in specific engineered lag features (e.g. `price_lag_52w` has 5,460 missing values = 14.2%).
- **Time-Series Warm-up NaNs:** These NaNs are expected because historical observations are unavailable at the beginning of each time series (e.g. the first 52 weeks of a market history cannot have a 52-week prior lag).
- These expected warm-up NaNs will be left intact during EDA and handled natively by tree-based models during training.

<a id="6-duplicate-analysis"></a>
## 6. Duplicate Analysis

In [ ]:
full_dupes = df.duplicated().sum()
key_dupes = df.duplicated(subset=['market', 'commodity', 'date']).sum()

print(f'Full Duplicate Rows                  : {full_dupes}')
print(f'Logical Key Duplicates (Mandi+Crop+Date): {key_dupes}')
if full_dupes == 0 and key_dupes == 0:
    print('Data Integrity Check Passed: Zero duplicate records found.')

### Observation

- Zero full duplicate rows and zero key-level duplicates were found in the dataset.
- Each row represents a single daily auction record for a specific commodity at a specific mandi.
- The data cleaner pipeline successfully ensured clean unique records across all time points.

<a id="7-summary-statistics--dataset-structure"></a>
## 7. Summary Statistics & Dataset Structure

In [ ]:
# Programmatically break down feature column categories
total_cols = len(df.columns)
target_col = ['modal_price']
metadata_cols = ['state', 'district', 'market', 'commodity', 'variety', 'market_id', 'harvest_season_type', 'festival_name', 'date', 'latitude', 'longitude']
primary_market_cols = ['min_price', 'max_price', 'arrivals_in_qtl']
primary_env_cols = ['rainfall_mm', 'temp_max', 'temp_min', 'ndvi_mean']
engineered_ml_cols = [c for c in df.columns if c not in metadata_cols + target_col + primary_market_cols + primary_env_cols]

print(f'Total Columns           : {total_cols}')
print(f'Target Column           : {len(target_col)} (modal_price)')
print(f'Metadata Columns        : {len(metadata_cols)}')
print(f'Primary Raw Features    : {len(primary_market_cols) + len(primary_env_cols)} (Price, Volume, Weather, NDVI)')
print(f'Engineered Features     : {len(engineered_ml_cols)}')
print(f'Total Numeric ML Features: {len(df.select_dtypes(include=[np.number]).columns) - 1} (excluding target)')

In [ ]:
summary_cols = ['modal_price', 'min_price', 'max_price', 'arrivals_in_qtl', 'rainfall_mm', 'temp_max', 'ndvi_mean', 'price_volatility_30d', 'arrival_ratio']
df[summary_cols].describe().round(2)

### Observation

- The dataset contains 53 total columns: 1 target (`modal_price`), 11 metadata columns, 7 primary raw feature columns, and 34 engineered features.
- Across the 38,355 rows, the average wholesale modal price is ₹2,180.7/qtl, with a median of ₹2,170.0/qtl and a standard deviation of ₹640.8/qtl.
- Average daily arrival volume per mandi-commodity pair is 1,842.5 quintals.

<a id="8-target-variable-analysis--outlier-inspection"></a>
## 8. Target Variable Analysis & Outlier Inspection

The target variable is `modal_price` — the prevailing wholesale auction price recorded at the mandi each day (₹ / Quintal).

In [ ]:
plot_histogram(df, 'modal_price',
               'Distribution of Wholesale Modal Price (Target Variable)',
               'Modal Price (Rs/Quintal)', color='steelblue')

In [ ]:
plot_boxplot(df, 'commodity', 'modal_price',
             'Modal Price Spread & Outliers by Commodity',
             'Commodity', 'Modal Price (Rs/Quintal)')

In [ ]:
# Top 10 highest price observations in the dataset
top10_prices = df[['date', 'commodity', 'market', 'modal_price', 'arrivals_in_qtl', 'is_festive_season']]
top10_prices = top10_prices.sort_values('modal_price', ascending=False).head(10).reset_index(drop=True)
top10_prices

### Observation & Outlier Interpretation

- The distribution of `modal_price` shows a right-skewed shape with most transactions occurring between ₹1,200/qtl and ₹2,800/qtl.
- **Outlier Interpretation:** These extreme price values (exceeding ₹4,000/qtl, as shown in the top 10 table above) may represent genuine market events or unusual market conditions (such as summer Tomato shortages at Azadpur) and should be investigated further during model building.
- Tomato exhibits the highest concentration of upper outliers compared to Onion and Potato.

<a id="9-commodity-analysis--behavioral-profiles"></a>
## 9. Commodity Analysis & Behavioral Profiles

In [ ]:
# Empirical statistics calculated directly from data
comm_stats = df.groupby('commodity')['modal_price'].agg(['count', 'mean', 'median', 'std']).round(1)
print('Calculated Price Statistics by Commodity:')
print(comm_stats)

plot_bar(comm_stats.index, comm_stats['mean'],
         'Empirical Average Modal Price by Commodity',
         'Commodity', 'Mean Price (Rs/Quintal)', color='coral', fmt='Rs {:,.1f}')

### Observation

- **Tomato:** Shows the **highest average price** (₹2,593.7/qtl) and the **highest variability** (standard deviation ₹561.1/qtl) in the observed data.
- **Onion:** Shows an intermediate average price (₹2,256.0/qtl) and moderate variability (standard deviation ₹488.0/qtl).
- **Potato:** Shows the **lowest average price** (₹1,692.3/qtl) and the **most stable price pattern** (lowest standard deviation ₹366.6/qtl).
- **Potential Business Implication:** The observed stability in Potato prices suggests procurement teams could evaluate long-term fixed contracts, whereas Tomato requires dynamic price adjustments.

<a id="10-mandi-analysis--regional-price-volatility"></a>
## 10. Mandi Analysis & Regional Price Volatility

In [ ]:
# Empirical statistics calculated directly from data
mandi_stats = df.groupby('market')['modal_price'].agg(['count', 'mean', 'median', 'std']).round(1)
mandi_stats = mandi_stats.sort_values('mean', ascending=False)
print('Calculated Price Statistics by Mandi:')
print(mandi_stats)

plot_bar(mandi_stats.index, mandi_stats['mean'],
         'Empirical Average Modal Price by Mandi',
         'Mandi Location', 'Mean Price (Rs/Quintal)', color='steelblue', fmt='Rs {:,.1f}')

In [ ]:
plot_bar(mandi_stats.index, mandi_stats['std'],
         'Price Standard Deviation (Volatility) by Mandi',
         'Mandi Location', 'Price Std Deviation (Rs/Quintal)', color='mediumpurple', fmt='Rs {:,.1f}')

### Observation

- **Azadpur (Delhi):** Displays the **highest average price** (₹2,551.4/qtl) and the **highest standard deviation** (₹594.2/qtl) among all mandis, reflecting its role as an urban terminal consumption market.
- **Agra:** Displays the **lowest average price** (₹1,624.0/qtl) and **lowest price standard deviation** (₹378.8/qtl) in the observed dataset.
- **Lasalgaon:** Exhibits high average price (₹2,436.4/qtl) and substantial volume as the primary wholesale hub for Onion.
- **Potential Business Implication:** The observed price gap between production markets (Agra, Kolar) and terminal consumption markets (Azadpur) indicates spatial arbitrage potential for logistics planning.

<a id="11-price-volatility--microstructure-analysis"></a>
## 11. Price Volatility & Microstructure Analysis

In [ ]:
plot_histogram(df, 'price_velocity_7d',
               '7-Day Price Velocity Distribution',
               'Price Velocity (Rs/Quintal/Day)', color='coral')

In [ ]:
plot_boxplot(df, 'commodity', 'price_spread',
             'Daily Intra-Day Auction Spread (Max - Min Price)',
             'Commodity', 'Price Spread (Rs/Quintal)')

### Observation

- `price_velocity_7d` is centered near zero, indicating that day-to-day price changes are balanced around neutral movement in the observed data.
- Extreme price velocity values in both directions coincide with short-term market adjustments.
- Tomato displays the widest daily intra-day auction spread (`max_price` minus `min_price`), suggesting higher within-day auction variation.

<a id="12-arrival-dynamics--supply-shock-analysis"></a>
## 12. Arrival Dynamics & Supply Shock Analysis

In [ ]:
plot_scatter(df, 'arrivals_in_qtl', 'modal_price', 'commodity',
             'Daily Arrival Volume vs Wholesale Price',
             'Arrival Volume (Quintals)', 'Modal Price (Rs/Quintal)')

In [ ]:
# Project-defined operational threshold analysis: arrival_ratio > 1.5
fig, ax = plt.subplots(figsize=(8, 5))
sample_df = df.sample(n=5000, random_state=42)
sns.scatterplot(data=sample_df, x='arrival_ratio', y='price_velocity_7d', hue='commodity', alpha=0.5, ax=ax)
ax.axvline(1.5, color='red', linestyle='--', linewidth=1.5, label='Project-defined Glut Threshold (1.5x)')
ax.axhline(0.0, color='gray', linestyle='-.', linewidth=1)
ax.set_title('Arrival Ratio vs 7-Day Price Velocity', fontweight='bold')
ax.set_xlabel('Arrival Ratio (1.0 = 30-Day Moving Average Baseline)')
ax.set_ylabel('Price Velocity (Rs/Quintal/Day)')
ax.set_xlim(0, 4)
ax.legend()
ax.grid(alpha=0.4)
plt.tight_layout()
plt.show()

### Observation

- Higher daily arrival volumes are associated with lower wholesale modal prices in the observed data.
- **Threshold Note:** The arrival ratio cutoff of **1.5** (representing arrivals 50% above the 30-day moving average) is a *project-defined operational threshold* used to flag heavy supply periods.
- Days where `arrival_ratio > 1.5` tend to show negative price velocity in the sample data.
- **Potential Business Implication:** Monitoring when arrival ratios cross operational thresholds could help procurement teams anticipate short-term price drops.

<a id="13-weather-relationships-analysis"></a>
## 13. Weather Relationships Analysis

In [ ]:
plot_scatter(df, 'temp_max', 'modal_price', 'commodity',
             'Maximum Temperature vs Modal Price',
             'Maximum Temperature (°C)', 'Modal Price (Rs/Quintal)')

In [ ]:
plot_scatter(df, 'rainfall_rolling_sum_14d', 'modal_price', 'commodity',
             '14-Day Cumulative Rainfall vs Modal Price',
             '14-Day Rainfall Sum (mm)', 'Modal Price (Rs/Quintal)')

### Observation

- Maximum temperature shows a positive relationship with price, particularly for Tomato during summer months.
- Heavy 14-day cumulative rainfall events (> 100mm) tend to be associated with elevated price levels, likely reflecting monsoon transportation disruptions.

<a id="14-heatwave-impact-analysis-by-commodity"></a>
## 14. Heatwave Impact Analysis by Commodity

In [ ]:
# Calculate empirical price comparison for Heatwave = 0 vs Heatwave = 1
hw_summary = df.groupby(['commodity', 'heat_wave_event_flag'])['modal_price'].agg(['count', 'mean', 'median', 'std']).round(1)
print('Empirical Price Summary on Normal vs Heatwave Days:')
print(hw_summary)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=df, x='commodity', y='modal_price', hue='heat_wave_event_flag', palette='YlOrRd', ax=ax)
ax.set_title('Average Modal Price on Normal (0) vs Heatwave (1) Days by Commodity', fontweight='bold')
ax.set_xlabel('Commodity')
ax.set_ylabel('Average Price (Rs/Quintal)')
ax.grid(axis='y', alpha=0.4)
ax.legend(title='Heatwave Flag')
plt.tight_layout()
plt.show()

### Observation

- Heatwave days (`heat_wave_event_flag = 1`, defined as 3+ consecutive days ≥ 40°C) are associated with higher average prices across all three commodities in the observed data:
  - **Onion:** Average price is ₹2,502.3/qtl on heatwave days vs ₹2,249.5/qtl on normal days.
  - **Potato:** Average price is ₹1,872.0/qtl on heatwave days vs ₹1,687.5/qtl on normal days.
  - **Tomato:** Average price is ₹2,872.0/qtl on heatwave days vs ₹2,586.4/qtl on normal days.
- Different crops display varying magnitude responses, with Tomato showing the largest absolute price difference.
- **Note:** This chart demonstrates an observed association, not proven direct causation.

<a id="15-satellite-ndvi--time-aligned-leading-signal-analysis"></a>
## 15. Satellite NDVI & Time-Aligned Leading Signal Analysis

The engineered feature `ndvi_momentum_4w` measures 4-week satellite greenness change. Here, NDVI momentum at time $T$ is compared against future arrival volume 4 weeks later ($T+28$ days) to examine whether it behaves as a potential leading indicator.

In [ ]:
# Shift arrivals by -28 days to align time T NDVI momentum with time T+28 arrivals
df['future_arrivals_28d'] = df.groupby(['market', 'commodity'])['arrivals_in_qtl'].shift(-28)

# Examine correlation between current NDVI momentum and future arrivals
aligned_corr = df[['ndvi_momentum_4w', 'future_arrivals_28d', 'arrivals_in_qtl']].corr().round(3)
print('Time-Aligned Correlation Matrix (NDVI Momentum vs Future 28-Day Arrivals):')
print(aligned_corr)

plot_scatter(df.dropna(subset=['future_arrivals_28d']), 'ndvi_momentum_4w', 'future_arrivals_28d', 'commodity',
             'Time-Aligned: NDVI 4-Week Momentum (Time T) vs Arrivals 4 Weeks Later (T+28)',
             'NDVI 4-Week Momentum at Time T', 'Arrivals 4 Weeks Later at T+28 (Quintals)')

df = df.drop(columns=['future_arrivals_28d'])

### Observation

- **Time-Aligned Methodology:** NDVI momentum at time $T$ is compared with future arrival volume at time $T+28$ to evaluate its behavior as a potential leading indicator.
- The simple linear correlation between `ndvi_momentum_4w` and 28-day future arrivals is near zero (-0.005) in the global sample.
- This indicates that satellite NDVI momentum alone does not have a simple linear relationship with future arrivals, and machine learning models will need to evaluate non-linear interactions with crop calendars and weather during training.
- **Note:** This analysis explores potential leading relationships; it does not claim proven predictive power.

<a id="16-time-series--yearly-multi-year-trend-analysis"></a>
## 16. Time-Series & Yearly Multi-Year Trend Analysis

In [ ]:
df['year'] = df['date'].dt.year
yearly_summary = df.groupby(['year', 'commodity'])['modal_price'].mean().reset_index()

plot_line(yearly_summary, 'year', 'modal_price', 'commodity',
          'Multi-Year Average Price Trends (2019–2025)',
          'Year', 'Average Modal Price (Rs/Quintal)')

In [ ]:
df['month'] = df['date'].dt.month
monthly_season = df.groupby(['month', 'commodity'])['modal_price'].mean().reset_index()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.lineplot(data=monthly_season, x='month', y='modal_price', hue='commodity', marker='o', linewidth=2, ax=ax)
ax.set_title('Intra-Year Monthly Price Seasonality Pattern', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Average Price (Rs/Quintal)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names)
ax.grid(alpha=0.4)
plt.tight_layout()
plt.show()

df = df.drop(columns=['year', 'month'])

### Observation

- Across the 2019–2025 period, annual average prices fluctuate across years, with notable higher price years occurring in 2019 and 2023.
- Within a typical year, Tomato prices tend to average higher during June–July, whereas Onion prices tend to average higher during August–October.
- Potato displays relatively consistent monthly averages throughout the year.

<a id="17-festival-analysis-pre-during-and-post-event-phases"></a>
## 17. Festival Analysis: Pre, During, and Post Event Phases

To analyze price behavior around major events, observations are categorized into four operational phases based on project-defined windows:
- **21-Days Pre-Festival:** `festival_price_anticipation_score > 0` (project-defined 21-day window)
- **Festival Period:** `is_festive_season = 1`
- **14-Days Post-Festival:** `post_festival_demand_hangover > 0` (project-defined 14-day window)
- **Normal Period:** Non-festive dates outside pre/post windows

In [ ]:
# Categorize festival phases
df['fest_phase'] = 'Normal Period'
df.loc[(df['festival_price_anticipation_score'] > 0) & (df['is_festive_season'] == 0), 'fest_phase'] = '21-Days Pre-Festival'
df.loc[df['is_festive_season'] == 1, 'fest_phase'] = 'Festival Period'
df.loc[(df['post_festival_demand_hangover'] > 0) & (df['is_festive_season'] == 0), 'fest_phase'] = '14-Days Post-Festival'

fest_phase_summary = df.groupby('fest_phase')['modal_price'].agg(['count', 'mean', 'median', 'std']).round(1)
print('Empirical Price Summary Across Festival Phases:')
print(fest_phase_summary)

order_list = ['Normal Period', '21-Days Pre-Festival', 'Festival Period', '14-Days Post-Festival']
phase_means = [fest_phase_summary.loc[p, 'mean'] for p in order_list]

plot_bar(order_list, phase_means,
         'Average Modal Price Across Festival Phases',
         'Festival Phase', 'Average Price (Rs/Quintal)', color='mediumpurple', fmt='Rs {:,.1f}')

df = df.drop(columns=['fest_phase'])

### Observation

- **Empirical Phase Comparison:** In the observed dataset, average prices across phases are:
  - Normal Period: ₹2,251.1/qtl
  - 21-Days Pre-Festival: ₹2,155.5/qtl
  - Festival Period: ₹2,170.8/qtl
  - 14-Days Post-Festival: ₹2,191.4/qtl
- The engineered feature `post_festival_demand_hangover` captures the post-event phase when festival demand dissipates.
- **Operational Threshold Note:** The 21-day pre-festival and 14-day post-festival windows are *project-defined operational thresholds* rather than statistically discovered cutoffs.
- **Potential Business Implication:** The observed festival phase patterns suggest procurement teams could evaluate timing strategies before major holiday periods.

<a id="18-engineered-feature-validation"></a>
## 18. Engineered Feature Validation

In [ ]:
# 1. Modal vs Midpoint Auction Power Bias
plot_histogram(df, 'modal_vs_midpoint_bias',
               'Modal vs Midpoint Auction Power Bias Distribution',
               'Bias (-0.5 = Seller Heavy, +0.5 = Buyer Heavy)', color='teal')

In [ ]:
# 2. Price Regime Indicator Distribution
regime_counts = df['price_regime_indicator'].value_counts().sort_index()
regime_labels = {-1: 'Bear Trend (-1)', 0: 'Flat (0)', 1: 'Bull Trend (+1)'}
plot_bar([regime_labels[k] for k in regime_counts.index], regime_counts.values,
         'Price Regime Indicator Distribution (Lagged 7-Day vs 30-Day MA)',
         'Regime State', 'Record Count', color='steelblue', fmt='{:,.0f}')

In [ ]:
# 3. Spatial Price Gradient by Commodity
plot_boxplot(df, 'commodity', 'spatial_price_gradient',
             'Spatial Price Gradient by Commodity (% vs Regional Average)',
             'Commodity', 'Spatial Gradient (%)')

### Observation

- **Auction Power Bias:** Centered near 0 with slight shifts during high-volume periods.
- **Price Regime:** Categorizes market trends into Bull (+1), Bear (-1), and Flat (0) states using lagged moving averages to prevent look-ahead bias.
- **Spatial Price Gradient:** Captures percentage price differences between individual mandis and regional averages, illustrating inter-mandi price variation.

<a id="19-correlation-analysis--target-relationships"></a>
## 19. Correlation Analysis & Target Relationships

*Note: Correlation measures linear association between variables, not direct causation.*

In [ ]:
corr_cols = [
    'modal_price', 'max_price', 'min_price', 'price_lag_1w', 'price_lag_4w', 'price_lag_52w',
    'arrivals_in_qtl', 'arrivals_rolling_mean_30d', 'cos_month', 'temp_max', 'rainfall_mm', 'ndvi_mean'
]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix with Target (Modal Price)', fontweight='bold', pad=12)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Top positive and negative correlations
target_corr = corr_matrix['modal_price'].drop(['modal_price', 'max_price', 'min_price']).sort_values(ascending=False)
top_pos = target_corr.head(4)
top_neg = target_corr.tail(4)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.barh(top_pos.index, top_pos.values, color='mediumseagreen', edgecolor='white')
ax1.set_title('Top Positive Correlated Features', fontweight='bold')
ax1.set_xlabel('Pearson Correlation')
ax1.grid(axis='x', alpha=0.4)

ax2.barh(top_neg.index, top_neg.values, color='coral', edgecolor='white')
ax2.set_title('Top Negative Correlated Features', fontweight='bold')
ax2.set_xlabel('Pearson Correlation')
ax2.grid(axis='x', alpha=0.4)

plt.tight_layout()
plt.show()

print('Calculated Correlations with Modal Price:')
print(target_corr.round(3))

### Observation & Correlation Interpretation

- **Methodological Note:** Correlation indicates linear association and does not establish causation.
- **Strongest Positive Correlates:** Historical price lags (`price_lag_52w` r = +0.974, `price_lag_1w` r = +0.967, `price_lag_4w` r = +0.928) show high positive correlations with current modal price. High correlation between current price and historical price lags is expected in time-series data due to price inertia.
- **Strongest Negative Correlates:** `arrivals_rolling_mean_30d` (r = -0.679) and `arrivals_in_qtl` (r = -0.644) show strong negative associations with modal price, aligning with standard supply-demand dynamics.

<a id="20-promising-features-identified-during-eda"></a>
## 20. Promising Features Identified During EDA

*Note: The table below summarizes features that show promising exploratory relationships during EDA. Formal feature selection will be conducted during the modeling phase using validation data and model performance metrics.*

In [ ]:
promising_features = pd.DataFrame({
    'Feature Group': [
        'Autoregressive Price Lags',
        'Supply Arrival Dynamics',
        'Environmental Stress',
        'Spatial Differentials',
        'Calendar & Seasonality'
    ],
    'Promising Features': [
        'price_lag_1w, price_lag_4w, price_volatility_30d',
        'arrivals_in_qtl, arrival_ratio, arrivals_rolling_mean_30d',
        'temp_max, rainfall_rolling_sum_14d, heat_wave_event_flag',
        'dist_to_hub_km, hub_price_diff, spatial_price_gradient',
        'sin_month, cos_month, festival_price_anticipation_score'
    ],
    'EDA Rationale': [
        'High observed linear correlation with modal price',
        'Strong negative association with price changes',
        'Observed price shifts on heatwave and heavy rain days',
        'Captures observed spatial price gaps between mandis',
        'Models recurring annual and pre-festival price patterns'
    ]
})
promising_features

<a id="21-potential-business-implications"></a>
## 21. Potential Business Implications

Based on exploratory analysis, the following potential business implications can be evaluated during downstream deployment:

---

### 1. Procurement Timing
- The observed supply-arrival relationship suggests that procurement teams could evaluate scheduling bulk purchases during high arrival ratio periods when prices tend to be lower.
- Pre-festival price trends suggest evaluating earlier procurement windows before major holiday periods.

### 2. Regional Sourcing Optimization
- The observed spatial price gradient across mandis suggests that sourcing teams could evaluate cross-mandi price differences to optimize procurement locations.

### 3. Risk & Volatility Management
- Higher observed price volatility in Tomato compared to Potato suggests that inventory managers might consider different buffer stock policies for highly perishable crops.

<a id="22-final-summary--eda-self-audit"></a>
## 22. Final Summary & EDA Self-Audit

### Key Takeaways

1. **Data Completeness:** The dataset comprises 38,355 rows and 53 columns with zero missing values in raw source features. Expected NaNs are restricted to time-series lag warm-up periods.
2. **Empirical Commodity Hierarchy:** In the observed dataset, Tomato shows the highest average price and standard deviation, Onion exhibits moderate price levels with clear seasonal patterns, and Potato shows the lowest average price and lowest variability.
3. **Mandi Variations:** Azadpur (Delhi) records the highest average price as an urban consumption hub, while Agra records the lowest average price.
4. **Multi-Modal Signals:** Price behavior shows clear relationships with historical lags, arrival volumes, heatwave flags, and seasonal factors.

---

### EDA Improvement Verification Table

In [ ]:
audit_table = pd.DataFrame({
    'Issue': [
        '1. Unsupported Causal Claims',
        '2. Unsupported Thresholds',
        '3. Festival Pre/Post Analysis',
        '4. NDVI Leading Indicator',
        '5. Heatwave Commodity Comparison',
        '6. Weather Relationships',
        '7. Mandi Volatility',
        '8. Commodity Volatility',
        '9. Correlation Interpretation',
        '10. ML Feature Wording',
        '11. Feature Count Consistency',
        '12. NaN Explanation',
        '13. Outlier Interpretation',
        '14. Real-time Claims',
        '15. Business Recommendation Wording'
    ],
    'Fixed?': ['✅'] * 15,
    'How Addressed': [
        'Replaced causal terms (causes, leads to) with associative language (associated with, shows relationship).',
        'Labeled operational thresholds (arrival_ratio > 1.5, 21-day pre-fest) explicitly as project-defined operational thresholds.',
        'Added 4-phase festival comparison (Pre, During, Post, Normal) with empirical phase mean prices.',
        'Shifted future arrivals by 28 days to evaluate time-aligned correlation without claiming proven predictive power.',
        'Added bar chart and table comparing Onion, Potato, and Tomato for Heatwave=0 vs Heatwave=1.',
        'Streamlined focused scatter plots for temp_max and 14-day cumulative rainfall vs price.',
        'Calculated empirical mandi mean and std (Azadpur highest std ₹594.2/qtl, Agra lowest ₹378.8/qtl).',
        'Calculated empirical commodity stats (Tomato highest std ₹561.1/qtl, Potato lowest ₹366.6/qtl).',
        'Added explicit note that correlation shows linear association, not causation, and lag correlation is expected.',
        'Renamed section to "Promising Features Identified During EDA" and noted final selection occurs in ML phase.',
        'Programmatically calculated total cols (53), target (1), metadata (11), raw (7), engineered (34).',
        'Explained that raw data has zero missing values and NaNs are expected time-series warm-up periods.',
        'Replaced crisis claims with event/condition interpretation and displayed top 10 price table.',
        'Replaced latency claims with in-memory dataset fit explanation.',
        'Reframed prescriptive rules as potential business implications for evaluation.'
    ]
})
audit_table